In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

In [2]:
# ==========================================
# 1. ENVIRONMENT & DATA LOADING
# ==========================================
cwd = os.getcwd()
parent = os.path.dirname(cwd)
# Fallback logic for directory structure
file_path_combined_data = os.path.join(parent, "Data", "combined_data.parquet")
if not os.path.exists(file_path_combined_data):
    file_path_combined_data = "combined_data.parquet"

print(f"Loading dataset from: {file_path_combined_data}")
# Using pyarrow/fastparquet engine via pandas or polars depending on environment
df = pd.read_parquet(file_path_combined_data)

# Drop 'Price_signal' to eliminate collinearity with Price_profile & Price_level
if "Price_signal" in df.columns:
    df = df.drop(columns=["Price_signal"])

# Filter to Phase 2 active participants
df_phase2 = df[
    (df["Participation_Phase"] == "Phase_2") & 
    (df["Participation_status_Phase2"] == "OK")
].copy()

print(f"Loaded Phase 2 rows: {len(df_phase2):,}")

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\combined_data.parquet
Loaded Phase 2 rows: 11,996,256


In [3]:
# ==========================================
# 2. TARGET ENGINEERING: FPI
# ==========================================
print("Engineering Flexibility Propensity Index (FPI)...")
df_phase2['Temp_Bin'] = (df_phase2['Temperature24'] / 5).round(0) * 5

# Baseline estimation on non-experiment days
df_baselines = (
    df_phase2[df_phase2["Experiment_price_NOK_kWh"].isna()]
    .groupby(["ID", "Hour", "Temp_Bin"])["Demand_kWh"]
    .mean()
    .reset_index(name="Baseline_kWh")
)

df_exp_days = pd.merge(
    df_phase2[df_phase2["Experiment_price_NOK_kWh"].notna()],
    df_baselines,
    on=["ID", "Hour", "Temp_Bin"],
    how="left"
)
df_exp_days['Baseline_kWh'] = df_exp_days['Baseline_kWh'].fillna(df_exp_days['Demand_kWh'].mean())

df_fpi_daily = (
    df_exp_days.groupby(["ID", "Date", "Region", "Price_profile", "Price_level"])
    .agg(
        Actual_Peak_Demand=("Demand_kWh", "sum"),
        Base_Peak_Demand=("Baseline_kWh", "sum")
    )
    .reset_index()
)
df_fpi_daily['Flexibility_Propensity_Index'] = (
    (df_fpi_daily['Base_Peak_Demand'] - df_fpi_daily['Actual_Peak_Demand']) / 
    (df_fpi_daily['Base_Peak_Demand'] + 1e-6) * 100
).clip(-50.0, 50.0)

Engineering Flexibility Propensity Index (FPI)...


In [7]:
# ==========================================
# 4. APPENDIX B: GENERATING TABLES & FIGURES
# ==========================================

# --- TABLE B1: Dataset Dimensionality & Observation Breakdown ---
table_b1 = (
    df
    .group_by(["Participation_Phase", "Participation_Experiment", "Control_Price_Phase2"])
    .agg(
        Total_Rows = pl.len(),
        Unique_Households = pl.col("ID").n_unique()
    )
    .sort(["Participation_Phase", "Participation_Experiment"])
)

print("\n=== Table B1: Dataset Dimensionality & Observation Breakdown ===")
print(table_b1)

AttributeError: 'DataFrame' object has no attribute 'group_by'

In [8]:
# ==========================================
# 3. FIGURE B2 FIX: DIURNAL LOAD PROFILE
# ==========================================
df_diurnal = (
    df_phase2.assign(Is_Experiment=df_phase2["Experiment_price_NOK_kWh"].notna())
    .groupby(["Hour", "Is_Experiment"])["Demand_kWh"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
palette_dict = {False: "#1b9e77", True: "#d95f02"}

sns.lineplot(
    data=df_diurnal, 
    x="Hour", 
    y="Demand_kWh", 
    hue="Is_Experiment", 
    palette=palette_dict,
    marker="o", 
    linewidth=2.5,
    ax=ax,
    legend=False # Disable automatic buggy legend
)

ax.set_title("Mean Diurnal Load Profile: Experiment Days vs. Non-Experiment Days")
ax.set_xlabel("Hour of Day (1–24)")
ax.set_ylabel("Mean Hourly Electricity Demand (kWh)")
ax.set_xticks(range(1, 25))

# Explicitly create correct legend handles with matching colors
legend_handles = [
    mlines.Line2D([], [], color="#1b9e77", marker='o', linewidth=2.5, label="Non-Experiment Baseline"),
    mlines.Line2D([], [], color="#d95f02", marker='o', linewidth=2.5, label="Active Price Experiment Day")
]
ax.legend(handles=legend_handles, title="Condition", loc="upper left")

fig_path_b2 = os.path.join(parent, "Data", "Figure_B2_Diurnal_Load_Profiles.png")
plt.savefig(fig_path_b2, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved fixed Figure B2 to: {fig_path_b2}")

Saved fixed Figure B2 to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_B2_Diurnal_Load_Profiles.png


In [6]:
# ==========================================
# 4. GENERATING DUAL GRAPH (FIGURE 2a & 2b: OSLO, P0_30)
# ==========================================
# Filter for Oslo region on experiment days where signal is P0_30 (30 NOK/kWh peak in hours 17-18)
# In combined_data, P0_30 corresponds to Price_profile == 'P0' and Price_level == 30 (or checking signal string if present, here profile/level)
df_oslo_p030 = df_phase2[
    (df_phase2["Region"] == "Oslo") & 
    (df_phase2["Price_profile"] == "P0") & 
    (df_phase2["Price_level"] == 30)
].copy()

# If specific subset is empty, fallback to general P0 / experiment days in Oslo for demonstration
if len(df_oslo_p030) == 0:
    df_oslo_p030 = df_phase2[(df_phase2["Region"] == "Oslo") & (df_phase2["Experiment_price_NOK_kWh"].notna())].copy()

# Separate Control vs Treatment
df_oslo_ctrl = df_oslo_p030[df_oslo_p030["Control_Price_Phase2"] == "Control_group"].groupby("Hour")["Demand_kWh"].mean().reset_index()
df_oslo_treat = df_oslo_p030[df_oslo_p030["Control_Price_Phase2"] != "Control_group"].groupby("Hour")["Demand_kWh"].mean().reset_index()

# Calculate mean difference for adjustment (Graph 2b math)
mean_diff = df_oslo_treat["Demand_kWh"].mean() - df_oslo_ctrl["Demand_kWh"].mean()
df_oslo_ctrl_adjusted = df_oslo_ctrl.copy()
df_oslo_ctrl_adjusted["Demand_kWh"] += mean_diff # Shift control curve to match treatment baseline

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Highlight peak hours (17 and 18)
for ax in axes:
    ax.axvspan(16.5, 18.5, color="gray", alpha=0.3, label="Peak Hours (17–18)")

# Graph A: Observed Data
axes[0].plot(df_oslo_ctrl["Hour"], df_oslo_ctrl["Demand_kWh"], marker='o', color="#333333", label="Control")
axes[0].plot(df_oslo_treat["Hour"], df_oslo_treat["Demand_kWh"], marker='o', color="#777777", label="Treatment")
axes[0].set_title("a) Observed Data")
axes[0].set_xlabel("Hour of the day")
axes[0].set_ylabel("Average demand (kWh/h)")
axes[0].set_xticks(range(2, 25, 2))
axes[0].set_ylim(2.2, 3.4)
axes[0].legend(loc="upper right")

# Graph B: Adjusted Data
axes[1].plot(df_oslo_ctrl_adjusted["Hour"], df_oslo_ctrl_adjusted["Demand_kWh"], marker='o', color="#333333", label="Control (Adjusted)")
axes[1].plot(df_oslo_treat["Hour"], df_oslo_treat["Demand_kWh"], marker='o', color="#777777", label="Treatment")
axes[1].set_title("b) Data Adjusted for Mean Differences")
axes[1].set_xlabel("Hour of the day")
axes[1].set_ylabel("Average demand (kWh/h)")
axes[1].set_xticks(range(2, 25, 2))
axes[1].set_ylim(2.2, 3.4)
axes[1].legend(loc="upper right")

fig_path_fig2 = os.path.join(parent, "Data", "Figure_2_Oslo_P0_30_Dual.png") if os.path.exists(os.path.join(parent, "Data")) else "Figure_2_Oslo_P0_30_Dual.png"
plt.savefig(fig_path_fig2, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved dual Figure 2 graph to: {fig_path_fig2}")

Saved dual Figure 2 graph to: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\Figure_2_Oslo_P0_30_Dual.png
